# 01 — Data Audit and EDA

**Goal:** audit the raw UCI Online Retail II data, establish its structure and data-quality issues, and perform the statistical analysis required by the project template.

**Important:** this notebook does not modify the raw workbook.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
# When notebooks are launched from notebooks/, move to repository root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw" / "online_retail_II.xlsx"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "figures"
MODELS = PROJECT_ROOT / "models"

DATA_PROCESSED.mkdir(exist_ok=True, parents=True)
FIGURES.mkdir(exist_ok=True, parents=True)
MODELS.mkdir(exist_ok=True, parents=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

assert DATA_RAW.exists(), f"Raw dataset not found: {DATA_RAW}"

xl = pd.ExcelFile(DATA_RAW)
print("Sheets:", xl.sheet_names)

frames = []
for sheet in xl.sheet_names:
    temp = pd.read_excel(DATA_RAW, sheet_name=sheet)
    temp["SourceSheet"] = sheet
    frames.append(temp)

df = pd.concat(frames, ignore_index=True)

print("Combined shape:", df.shape)
display(df.head())
display(df.tail())

## 1. Data structure and data types

In [ ]:
print(df.info())
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean()*100).round(2),
    "n_unique": df.nunique(dropna=True)
}).sort_values("missing_pct", ascending=False))

## 2. Basic quality audit

In [ ]:
duplicate_count = int(df.duplicated().sum())
negative_qty = int((df["Quantity"] < 0).sum())
zero_price = int((df["Price"] == 0).sum())
negative_price = int((df["Price"] < 0).sum())
missing_customer = int(df["Customer ID"].isna().sum())
missing_description = int(df["Description"].isna().sum())
cancellations = int(df["Invoice"].astype(str).str.startswith("C").sum())

audit = pd.DataFrame({
    "Issue": [
        "Exact duplicate rows",
        "Negative quantities",
        "Zero prices",
        "Negative prices",
        "Missing Customer ID",
        "Missing Description",
        "Cancellation invoices"
    ],
    "Count": [
        duplicate_count, negative_qty, zero_price, negative_price,
        missing_customer, missing_description, cancellations
    ]
})
audit["Percent of rows"] = (audit["Count"] / len(df) * 100).round(2)
display(audit)

## 3. Date range and coverage

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
print("Minimum date:", df["InvoiceDate"].min())
print("Maximum date:", df["InvoiceDate"].max())
print("Unique customers:", df["Customer ID"].nunique())
print("Countries:", df["Country"].nunique())

## 4. Descriptive statistics

In [ ]:
numeric_cols = ["Quantity", "Price"]
desc = df[numeric_cols].describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]).T
desc["skewness"] = df[numeric_cols].skew()
display(desc)

## 5. Transaction-value diagnostic

In [ ]:
eda = df.copy()
eda["TotalPrice"] = eda["Quantity"] * eda["Price"]

print("TotalPrice summary:")
display(eda["TotalPrice"].describe(percentiles=[0.01, 0.5, 0.75, 0.95, 0.99]).to_frame())

## 6. Relationships and visualizations

In [ ]:
# 6.1 Monthly transaction count
monthly = (
    df.assign(Month=df["InvoiceDate"].dt.to_period("M").astype(str))
      .groupby("Month").size()
)

plt.figure(figsize=(12, 5))
plt.plot(monthly.index, monthly.values, marker="o", linewidth=1)
plt.xticks(rotation=60, ha="right")
plt.title("Transaction Count by Month")
plt.xlabel("Month")
plt.ylabel("Transactions")
plt.tight_layout()
plt.savefig(FIGURES / "monthly_transaction_count.png", dpi=160)
plt.show()

In [ ]:
# 6.2 Quantity distribution on a log-like scale for readability
q = df.loc[df["Quantity"] > 0, "Quantity"]

plt.figure(figsize=(8, 5))
plt.hist(np.log1p(q), bins=60)
plt.title("Distribution of log(1 + Quantity) for Positive Purchases")
plt.xlabel("log(1 + Quantity)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(FIGURES / "quantity_log_distribution.png", dpi=160)
plt.show()

In [ ]:
# 6.3 Top countries by number of transactions
country_counts = df["Country"].value_counts().head(12)

plt.figure(figsize=(10, 5))
plt.bar(country_counts.index.astype(str), country_counts.values)
plt.xticks(rotation=45, ha="right")
plt.title("Top Countries by Transaction Count")
plt.xlabel("Country")
plt.ylabel("Transactions")
plt.tight_layout()
plt.savefig(FIGURES / "top_countries_transactions.png", dpi=160)
plt.show()

## 7. Key findings to carry forward

Write 3–5 evidence-based observations here after running the notebook. Examples of topics to discuss are:
- missing Customer IDs;
- duplicate records;
- cancellation/return transactions;
- zero or negative prices;
- strong skew in quantity/price;
- temporal concentration of sales.

**Do not invent the final statements before the notebook is run.**